# Football Analysis Modular Pipeline: Step-by-Step Processor Testing

This notebook demonstrates the modular football video analysis pipeline, testing each processor step by step with real video data and visualizing the results after each stage.

In [ ]:
# Import core libraries for data handling and visualization
import cv2
import matplotlib.pyplot as plt
from football_ai.domain.data_models import VideoData, FrameData

import os


## Load Real Data

We will load a sample football video, extract a few frames, and prepare the data for processing through the pipeline.

In [ ]:
# Path to a sample video (update if needed)
video_path = 'input_videos/08fd33_4.mp4'
assert os.path.exists(video_path), f"Video not found: {video_path}"

cap = cv2.VideoCapture(video_path)
frame_rate = cap.get(cv2.CAP_PROP_FPS)
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
duration = cap.get(cv2.CAP_PROP_FRAME_COUNT) / frame_rate

frames = []
for i in range(5):  # Load first 5 frames for demo
    ret, frame = cap.read()
    if not ret:
        break
    frames.append(FrameData(
        frame_number=i,
        timestamp=cap.get(cv2.CAP_PROP_POS_MSEC) / 1000.0,
        raw_frame=frame
    ))
cap.release()

video_data = VideoData(
    video_path=video_path,
    frame_rate=frame_rate,
    resolution=(width, height),
    duration=duration,
    frames=frames
)

print(f"Loaded {len(frames)} frames from {video_path}")
model_paths = {
    'detection': 'models/detect/best.pt',
}

## Visualize Results After Each Processor

After each processor, we will visualize the results for the first frame to observe the changes and outputs at each stage.

In [ ]:
# Object Detection
from football_ai.detection.object_detection_processor import ObjectDetectionProcessor
object_detection = ObjectDetectionProcessor(model_paths['detection'])
video_data = object_detection.process(video_data)



# Print the first data in the first frame after detection
print("First frame after object detection:")
print(video_data.frames[0])



In [ ]:
# tracking
from football_ai.tracking.track_processor import TrackProcessor
tracker = TrackProcessor()
video_data = tracker.process(video_data)

# Print the first data in the first frame after tracking
print("First frame after tracking:")
print(video_data.frames[0])


In [ ]:
# Object motion 
from football_ai.motion.object_motion_processor import ObjectMotionProcessor
object_motion_processor = ObjectMotionProcessor()
video_data = object_motion_processor.process(video_data)

# Print the first data in the first frame after object motion
print("First frame after object motion:")
print(video_data.frames[0])

In [ ]:
# Camera motion
from football_ai.motion.camera_motion_processor import CameraMotionProcessor
camera_motion_processor = CameraMotionProcessor()
video_data = camera_motion_processor.process(video_data)

# Print the first data in the first frame after camera motion
print("First frame after camera motion:")
print(video_data.frames[0])

In [ ]:
# field_transform
from football_ai.transformation.field_transformation_processor import FieldTransformationProcessor
field_transform = FieldTransformationProcessor()
video_data = field_transform.process(video_data)

# Print the first data in the first frame after field transformation
print("First frame after field transformation:")
print(video_data.frames[0])

In [ ]:
# team assignment
from football_ai.assignment.team_assignment_processor import TeamAssignmentProcessor
team_assignment_processor = TeamAssignmentProcessor()
video_data = team_assignment_processor.process(video_data)
# Print the first data in the first frame after team assignment
print("First frame after team assignment:")
print(video_data.frames[0])

In [ ]:
# ball assignment
from football_ai.assignment.ball_assignment_processor import BallAssignmentProcessor
ball_assignment_processor = BallAssignmentProcessor()
video_data = ball_assignment_processor.process(video_data)
# Print the first data in the first frame after ball assignment
print("First frame after ball assignment:")
print(video_data.frames[0])

In [ ]:
from football_ai.rendering.renderer_processor import RendererProcessor
renderer = RendererProcessor(output_path="outputs/videos/demo_output.mp4")
rendered_det = renderer.process(video_data)

# read output video
output_path = "outputs/videos/demo_output.mp4"
cap = cv2.VideoCapture(output_path)
cap.set(cv2.CAP_PROP_POS_FRAMES, 4)  # Reset to the first frame

ret, frame_det = cap.read()
cap.release()
if not ret:
    raise RuntimeError("Failed to read the output video frame.")
# Display the first frame after object detection    

plt.figure(figsize=(8, 5))
plt.imshow(cv2.cvtColor(frame_det, cv2.COLOR_BGR2RGB))
plt.title("Rendered after ObjectDetectionProcessor")
plt.axis('off')
plt.show()


